[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/00_how_the_data_came_in.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 00. How the data came in

**What this notebook does.** It walks through the whole data pipeline, from the DPG Alliance registry to the files in
`data/`, in ten numbered steps. For every step it says *which script ran*, *which outside source it talked to*,
*what went in*, *what came out* (and where that file sits in `data/`), *roughly how long it takes* and *which
METHODOLOGY decisions govern it* (the decision log is `METHODOLOGY.md (not published yet)`; ids look like D14).

**Why it exists.** The other five notebooks (01 to 05) start from the finished files in `data/`. This one shows how those
files were made, so that nothing in them is a black box.

**Three modes, one switch.** The first code cell below holds a single flag, `MODE`:

| `MODE` | what happens | needs |
|---|---|---|
| `"read"` (default) | Nothing touches the network. Cells that would fetch something print what they *would* do, and show the matching finished file from `data/` instead. | nothing |
| `"demo"` | Pulls **one small DPG** end to end: registry entry, its repository, GitHub collection for that one repository, then the aggregation scripts, ending in a quarterly row that is compared with the released one. Takes a few minutes. Writes **only** to the scratch folder `_demo_output/`, never to `data/`. | your own GitHub token |
| `"full"` | Re-runs the pipeline scripts in order for the **whole registry**. Takes many hours (GitHub allows 5,000 GraphQL points per hour) and a token. Also writes only to a scratch folder, `_full_run/`. A second switch (`CONFIRM_FULL_RUN`) must be set on purpose. | your own GitHub token, time |

**Your token.** The scripts read a GitHub personal access token (a classic token with public read access is enough) from the
environment variable `GITHUB_TOKEN`, or from a file called `.env` that *you* create next to the notebook or in `release/`,
containing one line `GITHUB_TOKEN=ghp_yourtoken`. The notebook never prints it and never stores it in its outputs.
`.env` is listed in `.gitignore`. Read mode needs no token.

**Reading order.** Part A is the numbered walk-through in plain language (markdown only). Part B repeats the steps with code:
each step has a cell that, depending on `MODE`, shows the finished file or really runs the script. Part C is the inventory of
every file in `data/`.

## Part A. The pipeline in ten steps

The scripts named below are in `extras/pipeline/`, unchanged. Each one finds its folders relative to its own location
(`ROOT = parent of the scripts folder`), reads inputs from `ROOT/data` and `ROOT/cache`, and writes outputs there. That is why
the demo and full modes copy the scripts into a scratch workspace first: the scripts then write into the scratch folder and
cannot touch `data/`.

### 1. The registry (DPG Alliance API)
* **Script / source.** No script: one HTTP GET to the public API `https://app.digitalpublicgoods.net/api/dpgs`. No key needed.
* **What it gives.** One JSON record per certified DPG: id (`GID...`), name, aliases, categories, sectors, SDGs, listed repositories,
  the owner the DPG declared (`clearOwnership`), and contact details of the people who registered it. 256 DPGs on 29 October 2026, all at stage "DPG".
* **Why this source.** The live API, not the archived `DPGAlliance/publicgoods-candidates` repository (archived in August 2024).
* **Output.** `data/raw/registry.json` in the project (not released: it contains contact names and e-mail addresses). The released
  extract is `data/registry_dpgs.csv` (id, name, stage, website).
* **Time.** Seconds. **Decisions.** D1 (live API, not the archive), D2 (window), D40 (what the registry looked like in the past).

### 2. Cleaning the registry into a DPG-to-repo map
* **Script.** `01_clean_registry.py`. **Input.** `data/raw/registry.json`.
* **What it does.** The `repositories[].url` field is messy: several URLs in one cell, links to LICENSE files, organisation-only URLs,
  typos such as a trailing dot. The script splits and normalises every URL to `owner/repo` and labels its type: `github` (a repo),
  `github_org` (only an organisation is listed) or `gitlab`. DPGs with no code link at all (data sets, content, standards) are written to a separate list.
* **Outputs (in `data/`).** `dpg_repo_map.csv`, `dpgs_without_code_repo.csv`. **Time.** Under a second. **Decisions.** D11 (URL cleaning), D10 (forks and mirrors).

### 3. Repo inventory and confidence tiers
* **Scripts.** `02a_inventory_repos.py` (GitHub REST API), then `02b_confidence_tiers.py` (no network).
* **What 02a does.** For every listed repo it asks GitHub whether the repo still resolves, whether it was renamed or moved (a redirect),
  and whether it is a fork, a mirror or archived. For every DPG that lists only an *organisation*, it enumerates every repository in that
  organisation (about 28,900 rows in total, of which about 27,200 are operating-system packaging repos of a single organisation). Raw pages are cached.
* **What 02b does.** Gives every repo a confidence tier and a reason. **high**: listed directly in the registry, resolves, not a fork or
  mirror, attributed to one DPG, or hand-verified. **medium**: picked by a rule for organisation-only DPGs. **low**: shared between DPGs.
  **excluded**: unresolved, fork/mirror, stale, or not matched. Only *high* feeds the headline results (the 166 DPGs and 190 repos).
* **Hand-curated decisions that live here.**
  (a) `manual_repo_review.csv`: a short list of repos verified or rejected by hand (`include`/`exclude` with a note), for example the case
  where a DPG's only listed repo is the Linux kernel tree. It overrides the fork rule. *This file is not part of the release*; its effect is visible in
  `all_repos.csv` as `confidence_reason = manual_verified` / `manual_exclude`.
  (b) **The D30 organisation rule**: for an organisation-only DPG, if the organisation's name contains the DPG's name (or an alias) *and*
  the organisation has at most 100 repos, every non-fork, non-stale, non-"docs/demo/website" repo in it is taken (tier medium). Otherwise only repos whose name matches the DPG are considered. Medium never enters the primary sample.
* **Outputs.** `all_repos.csv` (the released copy drops the 27,746 org repos rejected by the name rule and keeps a per-DPG count in
  `all_repos_not_matched_summary.csv`), `dpg_confidence.csv`. **Time.** About one second per listed repo plus a few minutes for the organisations.
  **Decisions.** D8, D10, D14, D30, D49 (the kernel-tree case).

### 4. GitHub collection (GraphQL)
* **Script.** `02_collect_github.py` (with `02c_size_repos.py` to pre-count PRs, issues and commits so small repos are collected first).
  **Source.** GitHub GraphQL API, authenticated. **Input.** `data/all_repos.csv`, tiers high and medium.
* **What is fetched per repo** (each one cached as `cache/<owner>__<repo>.json` *before* any computation, D12):
  repository metadata (created, pushed, archived, fork, mirror, stars, default branch, licence);
  **pull requests** (state, dates, author, who merged it, the first 10 reviews with reviewer and time, labels, additions/deletions/files changed, title and the first 3,000 characters of the body);
  **issues** (dates and state only); **releases** (dates, tag, pre-release flag); **tags** (with commit date); **commits** on the default branch since 1 January 2020 (date, headline, body, author name, e-mail and login), with a flag `ai_trailer` when the message carries an AI co-author trailer.
* **How it copes.** PRs and issues are fetched newest-updated first, and paging stops as soon as a page is last-updated before 2020 (this saves most of the cost on old, big repos). Commits use `since: 2020-01-01`. Releases and tags are read in full.
  A page that keeps failing (GitHub sometimes answers HTTP 200 with an unreadable body on big payloads) is retried with **half the page size** (down to 10). Progress is **checkpointed** every 20 pages in `cache/_partial/`, so a big repo resumes instead of restarting. Rate limits are waited out (the script sleeps until the reset time, including the secondary limit); server errors back off and retry. A repo already in the cache is skipped on re-run; delete its file to refetch.
* **Output.** The raw cache, 1.9 GB, **not released**. Everything downstream reads it. **Time.** Many hours for the registry (5,000 points per hour; a single large repo took about three minutes, a small one seconds). **Decisions.** D2, D12, D15 (PR body and commit text are kept), D6 (AI trailer flag).

### 5. Aggregation into quarterly panels
* **Script.** `03_aggregate.py` (reads the cache, `all_repos.csv`, `dpg_confidence.csv`; no network).
* **What it does.** Turns raw events into one row per repo per quarter and per DPG per quarter.
  **Bots and AI agents** are separated by account name patterns (D6): dependency/automation bots (Dependabot, Renovate, github-actions, a hand-checked list...) are counted in their own columns and excluded from headline counts; AI agents (Copilot coding agent, Devin, Claude, Codex, review bots such as CodeRabbit...) are their own class.
  **People and aliases**: team activity is a human commit author on the default branch, a human who merges a PR, or a human who reviews one (D19: PR *authorship* is not team activity, so team and output are never built from the same event). An e-mail seen with exactly one login is merged into that login; inside a DPG an unlinked e-mail is merged into a login if the author's name (two or more words) maps to exactly one login. A **core contributor** is active in at least 2 of the 3 months of a quarter (D4, D39).
  **Cohort rule** (D21, D45): output is summed over repos created before 2024-01-01, or with at least 50 commits dated before 2024 (migrated history), so new repos do not inflate output. People are the *union* across a DPG's repos, never summed. The partial current quarter is flagged and dropped from estimation (D20).
* **Outputs.** `repo_quarterly.csv`, `dpg_quarterly.csv`, `people_quarterly.csv` (hashed person ids only), `dpg_no_cohort_repo.csv`. **Time.** Minutes. **Decisions.** D3, D4, D5, D6, D9, D19, D20, D21, D39, D45.

### 6. Status classification
* **Script.** `03b_status.py` (reads the cache). **What it does.** A repo is **active**, **slowing** or **dormant** from events in the last 12 months before the fixed reference date 2026-10-29: merged PRs by humans or AI agents, plus published non-pre-release releases (tags if the repo has none). Dormant = none; slowing = fewer than half of the prior 12 months (with at least 6 then). A DPG takes its most active repo's status.
* **Outputs.** `repo_status.csv`, `dpg_status.csv`. **Time.** Minutes. **Decisions.** D32 (and D26, the dormancy question).

### 7. AI features: reading PR text and release notes
* **Scripts.** `03f_ai_feature_prs.py` (merged human PRs that *build* AI features, by keyword on title and cleaned body), `02e_release_notes.py` (fetches release notes from GitHub for DPGs with such PRs), `03g_release_ai.py` (did the change reach a release? and AI features named in release notes).
  Before matching, lines about AI *use* (co-author trailers, "generated with", tool names, disclosure sentences, template checkboxes) are stripped so that AI-assisted coding is not mistaken for an AI feature. Exploratory precision was about 85% on titles. `03e_ai_dependencies.py` (manifest scan for AI libraries), `03c_pr_text_ai.py` (the keyword term lists) belong to earlier or exploratory designs; their outputs are not in `data/`, except that 03h reuses the term lists in 03c.
* **Outputs.** `ai_feature_prs.csv`, `ai_feature_prs_shipped.csv`, `ai_feature_prs_yearly.csv`, `release_notes_ai.csv` (free-text `context` columns and nearly all PR titles are removed in the release). **Time.** Minutes each (they read the 1.9 GB cache; 02e makes API calls). **Decisions.** D36, D48, D46, D15, D16/D17 (superseded for reporting), D33 (what was parked).

### 8. Review classes and AI-disclosure
* **Scripts.** `13b_review_ai.py` and `03h_ai_disclosure.py`; both read the cache. The release distils their per-PR inputs into `pr_level.csv` (step 10).
* **Review class** of every merged human PR (13b): `human_reviewed` (a person other than the author reviewed it before merge), `ai_only` (no human reviewer, at least one AI agent reviewer), `automation_only` (only bots such as github-actions or mergify), `unreviewed`. Only the first 10 reviews per PR are collected, so `reviews_capped` marks PRs that may have more.
* **AI-disclosure** (03h): does the PR title or body say AI helped write the change? Lines that state AI was *not* used are dropped first (the negation filter, a logged deviation). Only the term group that passed the pre-set 80% precision check on a hand-checked sample, "Claude", is reported (83%); the other groups failed it. The hand verdicts are in `pr_text_ai_precision.csv`. Shares are a **lower bound**.
* **Outputs.** `pr_level.csv`, `review_bots_by_year.csv`, `pr_text_ai_precision.csv`, `qgis_template_check.csv`; the findings tables t7 and t8 are in `data/reference_outputs/`. **Time.** Minutes. **Decisions.** D15, D15a, D15b, D15d, D51, D6, D19.

### 9. Funding exposure
* **(a) Owner mapping.** `04_owner_clues.py` gathers clues per DPG (contact-e-mail and website domains, homepage title and copyright, the GitHub owner's profile, "developed by" phrases). The registry's own field `clearOwnership` (the owner the DPG declared to the Alliance) is the primary evidence (D38). A person then resolved each DPG to a legal entity, an owner type and a country: `data/dpg_owner_map.csv`. **Hand-curated**; a script that hard-coded the rebuilt table was withheld because it names an individual; its output is released with individuals' names replaced by `[individual]`. A second, independent coding checked agreement (D29 review).
* **(b) US awards.** `08_usaspending.py` (USAspending API: prime awards and sub-awards since FY2008, by recipient name and UEI) from a **hand-reviewed recipient map** built by `08a_build_recipient_map.py`; `08b_funding_flags.py` scores exposure 0 to 3 (3 = active foreign-aid prime award, 2 = foreign-aid sub-award, 1 = other federal money) on awards active from 2022-10-01 to 2025-01-19 (D17, DESIGN part B); `08c_apply_d29_outcomes.py` writes the second-coding outcomes; `08d_exposed_case_list.py` lists the exposed DPGs.
* **(c) Terminations and other donors.** `11_terminations.py` parses the ITPC list of USAID awards terminated as of 6 March 2025 (a leaked memo cleaned by a research group; unofficial); `11_join_terminations.py` joins them to the DPG awards by prime award id; `11_iati_collect.py` / `11_iati_match.py` search the IATI Datastore (needs an IATI key) for non-US donors; `11_donor_flags.py` writes the result (D24).
* **(d) Form 990s.** `05_ein_candidates.py` searches ProPublica Nonprofit Explorer for the EIN of US-nonprofit owners; a person picks the right one into `data/org_ein_map.csv` (**hand-curated**). `06_pull_990.py` pulls ProPublica's structured data (mostly ends 2023); `09_990_panel.py` reads the **IRS e-file XML** (TEOS) for the later years and for fields ProPublica lacks (government grants, employees). The small helper that downloads those zip files was **withheld for privacy** (it carried a personal contact address in its request header), so step (d) cannot be re-run from the release; the result is `org_990_panel.csv` (D47).
* **(e) The joint sample and the locked test.** `07_analysis_sample.py` applies the inclusion rules in `DESIGN.md (not published yet)` (every exclusion logged); `09_event_study_spec.py --lock` froze the event-study specification into `prereg_lock.json` *before* outcomes were looked at (D13, D28).
* **Outputs.** `dpg_owner_map.csv`, `org_ein_map.csv`, `funding_flags.csv`, `donor_flags.csv`, `exposed_dpgs_case_list.csv`, `org_990_panel.csv`, `org_990_panel_codebook.md`, `analysis_sample.csv`, `prereg_lock.json`. **Time.** Hours of API calls plus the human review. **Decisions.** D7, D17, D18 to D29, D24, D38, D41, D47.

### 10. The build script
* **Script.** `scripts/99_build_release.py` in the *project* (it is deliberately not copied into `pipeline/`). **Source.** The finished project files and, for the per-PR file, the raw cache.
* **What it does.** Copies the final datasets into `data/`; drops free-text columns (`context`) and blanks PR titles except the few the post quotes; replaces individuals' names in small tables by `[individual]`; writes the registry extract without contact details; reads the cache once to distil `pr_level.csv` (one row per merged PR: repo, DPG, dates, author class, review class, disclosure flag; no titles, bodies, logins or e-mails); writes the two small evidence files; copies the pipeline scripts, minus the withheld ones (one with a personal contact address, one that names an individual, an internal message helper and a git helper); copies the findings tables and the docs; refreshes `numbers_ledger.csv`; and finishes with a privacy audit (no e-mail addresses, no contact names). Hashed person ids can be re-hashed with a private salt (`RELEASE_PEOPLE_SALT`).
* **Outputs.** All of `data/`, `pipeline/`, `data/reference_outputs/`, `docs/`. **Time.** Minutes (5 to 10 with the cache pass). **Decisions.** The privacy rules are listed in the script's header; D14 and D45 define what is kept.

## Part B. The steps with code

### The switch and the safety rules

Change `MODE` here and nowhere else. Everything else in the notebook follows from it.

In [2]:
# ======================================================================================================
#  THE ONE SWITCH
# ======================================================================================================
MODE = "read"        # "read"  : default. No network, no files written. Shows what each step would do and the finished data.
                     # "demo"  : one small DPG end to end (a few minutes); writes only to _demo_output/
                     # "full"  : the whole pipeline (many hours); writes only to _full_run/

# --- demo settings -------------------------------------------------------------------------------
DEMO_DPG_ID = "GID0090256"   # Deon (DrivenData): one listed repository (drivendataorg/deon), high confidence,
                             # ~50 merged PRs since 2020, so the GitHub collection takes seconds.
CLEAN_DEMO_AFTER = False     # True: delete _demo_output/ at the end of the notebook.

# --- full-mode safety switch ---------------------------------------------------------------------
CONFIRM_FULL_RUN = False     # MODE="full" does nothing unless this is also True. It will use your token for many hours.
FULL_WORKERS = 4             # parallel repos in the collector (02_collect_github.py --workers)

assert MODE in ("read", "demo", "full"), "MODE must be 'read', 'demo' or 'full'"
print("MODE =", MODE)

MODE = read


In [3]:
# ======================================================================================================
#  SETUP: paths, token handling, and the helper that runs a pipeline script
# ======================================================================================================
import json, os, re, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

# This notebook lives in the repository root. REL is the repository folder, DATA the finished datasets.
REL = Path(".").resolve()
if not (REL / "data").exists():                      # started from another folder? try the current one
    REL = Path.cwd().resolve()
assert (REL / "data").exists() and (REL / "extras" / "pipeline").exists(), f"cannot find data and extras/pipeline from {Path.cwd()}"
DATA, PIPE = REL / "data", REL / "extras" / "pipeline"

# The scratch workspace. The pipeline scripts find their folders relative to THEMSELVES (ROOT = parent of the
# 'scripts' folder), so we copy the scripts into WORK/scripts and run them there: they then read and write
# WORK/data and WORK/cache and cannot reach data. (This is the whole trick that keeps the release safe.)
WORK = {"read": None, "demo": REL / "_demo_output", "full": REL / "_full_run"}[MODE]
RUN_FULL = (MODE == "full" and CONFIRM_FULL_RUN)
if MODE == "full" and not CONFIRM_FULL_RUN:
    print("MODE='full' but CONFIRM_FULL_RUN is False: this notebook will only PRINT the plan. "
          "Set CONFIRM_FULL_RUN = True above to run it for real.")
LIVE = MODE == "demo" or RUN_FULL                    # True only when something really runs

def inside_scratch(p):
    # every write goes through this check: the path must be under WORK and must not be data
    p = Path(p).resolve()
    return WORK is not None and WORK.resolve() in p.parents and DATA.resolve() not in (p, *p.parents)

def say(msg):
    # used by cells that would fetch or run something: in read mode they only announce it
    print(("[read mode] would: " if not LIVE else "[run] ") + msg)

# ---- token: from the environment, else from a .env file YOU created. Never printed. ----
def get_token():
    t = os.environ.get("GITHUB_TOKEN")
    if t: return t.strip()
    for d in (Path.cwd(), REL, REL.parent):
        f = d / ".env"
        if f.exists():
            for line in f.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("GITHUB_TOKEN="):
                    return line.split("=", 1)[1].strip().strip("\"'")
    if "google.colab" in sys.modules:                    # Colab has no .env file: ask, without echo
        import getpass
        t = getpass.getpass("GitHub token (input is hidden, never saved or printed; leave empty to skip): ")
        return t.strip() or None
    return None

TOKEN = get_token() if LIVE else None
if LIVE and not TOKEN:
    raise SystemExit("No GITHUB_TOKEN found. Set the environment variable, or create a .env file with the line GITHUB_TOKEN=... "
                     "(see the notes at the top of this notebook). Read mode needs no token.")
print("token available:", bool(TOKEN) if LIVE else "not needed in read mode")   # the value itself is never shown

# ---- the workspace: copy the scripts there ----
def prepare_workspace(scripts):
    assert inside_scratch(WORK / "x")
    if MODE == "demo" and WORK.exists():
        assert WORK.name == "_demo_output"; shutil.rmtree(WORK)       # a demo always starts clean
    for sub in ("scripts", "data/raw", "cache", "docs/post/findings_tables"):
        (WORK / sub).mkdir(parents=True, exist_ok=True)
    for s in scripts:
        shutil.copy2(PIPE / s, WORK / "scripts" / s)
    env = WORK / ".env"            # 02a and 02e read a .env file themselves, so one has to exist.
    if not env.exists():           # Demo: an EMPTY file (the token reaches the scripts through the environment instead).
        src = next((d / ".env" for d in (Path.cwd(), REL, REL.parent) if (d / ".env").exists()), None)
        if MODE == "full" and src: shutil.copy2(src, env)             # full mode: 02e needs the token in the file
        else: env.write_text("")

def run_script(name, *args, tail=12, timeout=None):
    # Runs  python -I scripts/<name> <args>  inside the workspace (exactly how the project runs them) and prints the end of its output.
    cmd = [sys.executable, "-I", str(Path("scripts") / name), *args]
    if not LIVE:
        say("run  " + " ".join(["python", "-I", "scripts/" + name, *args]) + "   (inside the scratch workspace)")
        return None
    env = dict(os.environ, GITHUB_TOKEN=TOKEN, PYTHONIOENCODING="utf-8")
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    lines = []
    for ln in p.stdout:
        ln = ln.rstrip().replace(TOKEN, "***")           # belt and braces: the token can never reach the output
        lines.append(ln)
        if MODE == "full": print("   " + ln[:200], flush=True)
    p.wait()
    print(f"[run] python -I scripts/{name} {' '.join(args)}   -> exit {p.returncode}, {time.time()-t0:.1f}s")
    if MODE != "full":
        for ln in lines[-tail:]: print("   " + ln[:200])
    if p.returncode != 0: print("   !! the script failed; see the lines above")
    return p.returncode

print("repository folder:", REL)
print("scratch workspace:", WORK if WORK else "(none in read mode)")

token available: not needed in read mode
repository folder: <repository folder>
scratch workspace: (none in read mode)


### The pipeline at a glance

This table is the numbered walk-through of Part A in one place, with the files each step leaves in `data/`.

In [4]:
OVERVIEW = pd.DataFrame([
 (1,  "DPGA registry API",             "(inline HTTP GET)",                          "app.digitalpublicgoods.net/api/dpgs",   "-",                              "registry_dpgs.csv (extract)",                                    "seconds",            "D1, D2, D40"),
 (2,  "Clean registry -> repo map",    "01_clean_registry.py",                       "-",                                     "data/raw/registry.json",         "dpg_repo_map.csv, dpgs_without_code_repo.csv",                   "seconds",            "D10, D11"),
 (3,  "Repo inventory + confidence",   "02a_inventory_repos.py, 02b_confidence_tiers.py", "GitHub REST API",                  "dpg_repo_map.csv, [manual_repo_review.csv]", "all_repos.csv, all_repos_not_matched_summary.csv, dpg_confidence.csv", "minutes",   "D8, D10, D14, D30, D49"),
 (4,  "GitHub GraphQL collection",     "02c_size_repos.py, 02_collect_github.py",    "GitHub GraphQL API",                    "all_repos.csv",                  "(raw cache, 1.9 GB, not released)",                              "many hours",         "D2, D6, D12, D15"),
 (5,  "Quarterly panels",              "03_aggregate.py",                            "-",                                     "cache, all_repos, dpg_confidence", "repo_quarterly.csv, dpg_quarterly.csv, people_quarterly.csv, dpg_no_cohort_repo.csv", "minutes", "D3-D6, D9, D19-D21, D39, D45"),
 (6,  "Status classification",         "03b_status.py",                              "-",                                     "cache, all_repos",               "repo_status.csv, dpg_status.csv",                                "minutes",            "D32, D26"),
 (7,  "AI features in PRs / releases", "03f_ai_feature_prs.py, 02e_release_notes.py, 03g_release_ai.py", "GitHub REST (release notes)", "cache, all_repos",       "ai_feature_prs.csv, ai_feature_prs_shipped.csv, ai_feature_prs_yearly.csv, release_notes_ai.csv", "minutes", "D36, D46, D48"),
 (8,  "Review classes + AI disclosure","13b_review_ai.py, 03h_ai_disclosure.py",     "-",                                     "cache, all_repos",               "pr_level.csv, review_bots_by_year.csv, pr_text_ai_precision.csv, qgis_template_check.csv", "minutes", "D15, D15d, D51, D19"),
 (9,  "Funding exposure",              "04, 05, 06, 07, 08, 08a-d, 09_990_panel, 09_event_study_spec, 11_*", "ProPublica, USAspending, IATI, IRS e-file, ITPC list", "owner clues, hand-curated maps", "dpg_owner_map.csv, org_ein_map.csv, funding_flags.csv, donor_flags.csv, exposed_dpgs_case_list.csv, org_990_panel.csv, analysis_sample.csv, prereg_lock.json", "hours + human review", "D7, D17, D18-D29, D38, D47"),
 (10, "Build the release",             "scripts/99_build_release.py (project only)", "-",                                     "all of the above + cache",       "everything in data/",                                    "minutes",            "privacy rules in the script header"),
], columns=["step", "what", "script(s)", "outside source", "main inputs", "outputs (in data/ unless noted)", "time", "METHODOLOGY"])
OVERVIEW.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

step,what,script(s),outside source,main inputs,outputs (in data/ unless noted),time,METHODOLOGY
1,DPGA registry API,(inline HTTP GET),app.digitalpublicgoods.net/api/dpgs,-,registry_dpgs.csv (extract),seconds,"D1, D2, D40"
2,Clean registry -> repo map,01_clean_registry.py,-,data/raw/registry.json,"dpg_repo_map.csv, dpgs_without_code_repo.csv",seconds,"D10, D11"
3,Repo inventory + confidence,"02a_inventory_repos.py, 02b_confidence_tiers.py",GitHub REST API,"dpg_repo_map.csv, [manual_repo_review.csv]","all_repos.csv, all_repos_not_matched_summary.csv, dpg_confidence.csv",minutes,"D8, D10, D14, D30, D49"
4,GitHub GraphQL collection,"02c_size_repos.py, 02_collect_github.py",GitHub GraphQL API,all_repos.csv,"(raw cache, 1.9 GB, not released)",many hours,"D2, D6, D12, D15"
5,Quarterly panels,03_aggregate.py,-,"cache, all_repos, dpg_confidence","repo_quarterly.csv, dpg_quarterly.csv, people_quarterly.csv, dpg_no_cohort_repo.csv",minutes,"D3-D6, D9, D19-D21, D39, D45"
6,Status classification,03b_status.py,-,"cache, all_repos","repo_status.csv, dpg_status.csv",minutes,"D32, D26"
7,AI features in PRs / releases,"03f_ai_feature_prs.py, 02e_release_notes.py, 03g_release_ai.py",GitHub REST (release notes),"cache, all_repos","ai_feature_prs.csv, ai_feature_prs_shipped.csv, ai_feature_prs_yearly.csv, release_notes_ai.csv",minutes,"D36, D46, D48"
8,Review classes + AI disclosure,"13b_review_ai.py, 03h_ai_disclosure.py",-,"cache, all_repos","pr_level.csv, review_bots_by_year.csv, pr_text_ai_precision.csv, qgis_template_check.csv",minutes,"D15, D15d, D51, D19"
9,Funding exposure,"04, 05, 06, 07, 08, 08a-d, 09_990_panel, 09_event_study_spec, 11_*","ProPublica, USAspending, IATI, IRS e-file, ITPC list","owner clues, hand-curated maps","dpg_owner_map.csv, org_ein_map.csv, funding_flags.csv, donor_flags.csv, exposed_dpgs_case_list.csv, org_990_panel.csv, analysis_sample.csv, prereg_lock.json",hours + human review,"D7, D17, D18-D29, D38, D47"
10,Build the release,scripts/99_build_release.py (project only),-,all of the above + cache,everything in data/,minutes,privacy rules in the script header


### Step 1. The registry

In read mode we show the released extract. In demo/full mode we do the real GET. The full registry record holds contact names and
e-mail addresses of the people who registered each DPG, so **the notebook never prints those fields**; in demo mode only the one
chosen DPG's record is kept, and only inside the scratch folder.

In [5]:
REGISTRY_URL = "https://app.digitalpublicgoods.net/api/dpgs"      # public, no key

if not LIVE:
    say(f"GET {REGISTRY_URL}  and save the JSON to <workspace>/data/raw/registry.json")
    reg_extract = pd.read_csv(DATA / "registry_dpgs.csv")
    print(f"\nFinished file instead: data/registry_dpgs.csv has {len(reg_extract)} DPGs. stage counts: {reg_extract.stage.value_counts().to_dict()}")
    display(reg_extract.head(5))
else:
    import requests
    ALL_SCRIPTS = ["01_clean_registry.py", "02a_inventory_repos.py", "02b_confidence_tiers.py", "02_collect_github.py", "03_aggregate.py",
                   "03b_status.py", "03c_pr_text_ai.py", "03f_ai_feature_prs.py", "03h_ai_disclosure.py", "13b_review_ai.py"]
    if RUN_FULL:
        ALL_SCRIPTS += ["02c_size_repos.py", "02e_release_notes.py", "03g_release_ai.py", "04_owner_clues.py", "05_ein_candidates.py", "06_pull_990.py"]
    prepare_workspace(ALL_SCRIPTS)
    r = requests.get(REGISTRY_URL, timeout=120); r.raise_for_status()
    registry = r.json()
    print(f"live registry: {len(registry)} DPGs, stages {pd.Series([d.get('stage') for d in registry]).value_counts().to_dict()}  "
          f"(the release snapshot of 29 Oct 2026 has 256)")
    if MODE == "demo":
        chosen = [d for d in registry if d["dpgid"] == DEMO_DPG_ID]
        assert chosen, f"{DEMO_DPG_ID} not found in the live registry"
        to_save = chosen                                              # ONE record only
        print("demo DPG:", chosen[0]["name"], "| listed repositories:", [x.get("url") for x in chosen[0].get("repositories") or []])
        print("declared owner (clearOwnership, step 9a):", (chosen[0].get("clearOwnership") or [{}])[0].get("clearOwnershipName"))
    else:
        to_save = registry
    assert inside_scratch(WORK / "data/raw/registry.json")
    (WORK / "data/raw/registry.json").write_text(json.dumps(to_save), encoding="utf-8")

[read mode] would: GET https://app.digitalpublicgoods.net/api/dpgs  and save the JSON to <workspace>/data/raw/registry.json

Finished file instead: data/registry_dpgs.csv has 256 DPGs. stage counts: {'DPG': 256}


,dpgid,name,stage,website,n_listed_repositories
0,GID0090024,DIGIT,DPG,https://www.digit.org/,1
1,GID0090077,OpenCRVS,DPG,https://www.opencrvs.org/,1
2,GID0090283,Product Catalog Management Tool,DPG,http://productcatalog.io,1
3,GID0090284,Everwell Hub,DPG,https://www.everwell.org/,1
4,GID0090285,iHRIS,DPG,https://www.ihris.org/,1


### Step 2. Cleaning the registry into a DPG-to-repo map (`01_clean_registry.py`)

In [6]:
rc = run_script("01_clean_registry.py")
if not LIVE:
    rmap = pd.read_csv(DATA / "dpg_repo_map.csv")
    print("\nFinished file instead: data/dpg_repo_map.csv")
    print("rows:", len(rmap), "| DPGs per link type:", rmap.groupby("host").dpgid.nunique().to_dict())
    display(rmap[["dpgid", "dpg", "host", "raw_url", "full_name"]].head(5))
    print("DPGs with no code repository (dpgs_without_code_repo.csv):", pd.read_csv(DATA / "dpgs_without_code_repo.csv").dpgid.nunique())
else:
    display(pd.read_csv(WORK / "data/dpg_repo_map.csv")[["dpgid", "dpg", "host", "raw_url", "full_name"]])

[read mode] would: run  python -I scripts/01_clean_registry.py   (inside the scratch workspace)

Finished file instead: data/dpg_repo_map.csv
rows: 240 | DPGs per link type: {'github': 173, 'github_org': 21, 'gitlab': 9}


,dpgid,dpg,host,raw_url,full_name
0,GID0090024,DIGIT,github,https://github.com/egovernments/DIGIT-OSS,egovernments/DIGIT-OSS
1,GID0090077,OpenCRVS,github,https://github.com/opencrvs/opencrvs-core,opencrvs/opencrvs-core
2,GID0090283,Product Catalog Management Tool,gitlab,https://gitlab.com/pcmt/pcmt,pcmt/pcmt
3,GID0090284,Everwell Hub,gitlab,https://gitlab.com/everwell/hub-foss,everwell/hub-foss
4,GID0090285,iHRIS,github,https://github.com/iHRIS/iHRIS,iHRIS/iHRIS


DPGs with no code repository (dpgs_without_code_repo.csv): 54


### Step 3. Repo inventory and confidence tiers (`02a_inventory_repos.py`, `02b_confidence_tiers.py`)

Hand-curated decisions that enter here: the picks in `manual_repo_review.csv` (not released; visible as `manual_verified` /
`manual_exclude` below) and the **D30 organisation rule** (visible as `org_dedicated_rule_D30`). In demo mode the demo DPG has no
hand decision and its single listed repo is simply `high / listed_in_registry`. (One caveat of the single-DPG demo: the "shared between DPGs" check
can only see this one DPG.)

In [7]:
if not LIVE:
    say("run 02a_inventory_repos.py (GitHub REST: does each listed repo resolve? renamed? fork? archived? plus every repo of listed organisations)")
    say("run 02b_confidence_tiers.py (no network)")
    ar = pd.read_csv(DATA / "all_repos.csv", low_memory=False)
    print("\nFinished file instead: data/all_repos.csv, tier x reason:")
    display(ar.groupby(["confidence", "confidence_reason"]).size().rename("repos").reset_index())
    dc = pd.read_csv(DATA / "dpg_confidence.csv")
    print("DPG-level tier (dpg_confidence.csv):", dc.dpg_confidence.value_counts().to_dict())
else:
    run_script("02a_inventory_repos.py")
    run_script("02b_confidence_tiers.py")
    display(pd.read_csv(WORK / "data/all_repos.csv")[["dpg", "source", "full_name", "resolved", "moved", "fork", "mirror", "archived", "confidence", "confidence_reason"]])

[read mode] would: run 02a_inventory_repos.py (GitHub REST: does each listed repo resolve? renamed? fork? archived? plus every repo of listed organisations)
[read mode] would: run 02b_confidence_tiers.py (no network)

Finished file instead: data/all_repos.csv, tier x reason:


,confidence,confidence_reason,repos
0,excluded,fork_or_mirror,207
1,excluded,manual_exclude,2
2,excluded,org_repo_nonproduct_name,26
3,excluded,org_repo_stale_since_2020,33
4,excluded,unresolved,3
5,high,listed_in_registry,196
6,high,manual_verified,2
7,low,shared_between_dpgs,2
8,medium,org_dedicated_rule_D30,402
9,medium,org_name_match_unreviewed,297


DPG-level tier (dpg_confidence.csv): {'high': 166, 'medium': 23, 'no_usable_repo': 11, 'low': 2}


### Step 4. GitHub collection (`02_collect_github.py`)

First the actual GraphQL field lists, read from the script's source (so this is exactly what is requested), then the run.
In demo mode the script is given `--repos owner/name`, which makes it collect that single repository. In full mode it takes all high and
medium repos from `all_repos.csv`, smallest first (`02c_size_repos.py` supplies the size estimate), `FULL_WORKERS` repos at a time.

In [8]:
src = (PIPE / "02_collect_github.py").read_text(encoding="utf-8")
print("What the collector asks GitHub for (copied from 02_collect_github.py):\n")
for key in ("SINCE", "BODY_CAP", "AUTH", "PR", "ISSUE", "REL"):
    m = re.search(rf"^{key} = .*?(?=\n[A-Za-z_]+ = |\ndef |\Z)", src, re.S | re.M)
    if m: print(m.group(0).strip(), "\n")
print("Safeguards in the source: page-size halving ->", "page_size // 2" in src, "| checkpoint every 20 pages ->", "pages % 20 == 0" in src,
      "| early stop on old updatedAt ->", "stop_before" in src, "| waits for rate-limit reset ->", "x-ratelimit-reset" in src)

What the collector asks GitHub for (copied from 02_collect_github.py):

SINCE = "2020-01-01T00:00:00Z" 

BODY_CAP = 3000  # chars of PR bodyText kept 

AUTH = "author{login __typename}" 

PR = ("updatedAt createdAt mergedAt closedAt state title bodyText isDraft additions deletions changedFiles "
      "mergedBy{login} reviews(first:10){nodes{submittedAt author{login}}} labels(first:10){nodes{name}} " + AUTH) 

ISSUE = "updatedAt createdAt closedAt state" 

REL = "createdAt publishedAt tagName isPrerelease" 

Safeguards in the source: page-size halving -> True | checkpoint every 20 pages -> True | early stop on old updatedAt -> True | waits for rate-limit reset -> True


In [9]:
DEMO_REPO = None
if LIVE and MODE == "demo":
    DEMO_REPO = pd.read_csv(WORK / "data/all_repos.csv").query("confidence == 'high'").full_name.iloc[0]
    run_script("02_collect_github.py", "--repos", DEMO_REPO, "--workers", "1")
elif RUN_FULL:
    run_script("02c_size_repos.py")
    run_script("02_collect_github.py", "--workers", str(FULL_WORKERS))
else:
    say("run 02_collect_github.py --repos <owner/name>   (demo)  or  02_collect_github.py   (full: ~600 high+medium repos, many hours)")
    print("\nThe raw cache (cache/<owner>__<repo>.json, 1.9 GB) is not part of the release; the next steps' outputs are.")

# What the cache file for the demo repo holds (counts only; no names, titles or logins are printed):
if DEMO_REPO:
    f = WORK / "cache" / (DEMO_REPO.replace("/", "__") + ".json")
    d = json.loads(f.read_text(encoding="utf-8"))
    print(f"\ncache file {f.name}: {f.stat().st_size/1e6:.2f} MB, status={d['status']}")
    for k in ("prs", "issues", "releases", "tags", "commits"):
        print(f"  {k:9s}{len(d[k]):6d} items")
    print("  PR fields stored:", sorted(d["prs"][0].keys()))
    print("  commit fields stored:", sorted(d["commits"][0].keys()))
    print("  commits flagged ai_trailer:", sum(c.get("ai_trailer", False) for c in d["commits"]))

[read mode] would: run 02_collect_github.py --repos <owner/name>   (demo)  or  02_collect_github.py   (full: ~600 high+medium repos, many hours)

The raw cache (cache/<owner>__<repo>.json, 1.9 GB) is not part of the release; the next steps' outputs are.


### Step 5. Quarterly panels (`03_aggregate.py`)

The bot, AI-agent and cohort rules are plain constants at the top of the script. We print them so you can read the actual patterns.

In [10]:
agg_src = (PIPE / "03_aggregate.py").read_text(encoding="utf-8")
a = agg_src.index("COHORT_BEFORE"); b = agg_src.index("def actor_class"); c = agg_src.index("\n\n", b)
print(agg_src[a:c])

COHORT_BEFORE = pd.Timestamp("2024-01-01")
MIGRATION_MIN_COMMITS = 50  # D45: a repo created later still counts as cohort if it has >=50 commits dated before 2024 (migrated history)
TODAY = pd.Timestamp.today().normalize()
CURRENT_Q = str(TODAY.to_period("Q"))

# Automation accounts. Patterns catch "-bot", "[bot]", "-ci", "automation", release/builder bots; HAND_BOTS lists user accounts found by scanning the cache
# (checked, 2026-10-29). Names that merely contain "bot"/"robot" but belong to people (TimidRobot, vbothe23, ...) are deliberately NOT matched.
DEP_BOTS = re.compile(r"dependabot|renovate|github-actions|pre-commit-ci|snyk|greenkeeper|allcontributors|weblate|transifex|crowdin|codecov|imgbot|sonar"
                      r"|[-_.]bot(?:$|[-_.\[])|\[bot\]|^bot[-_]|[-_]ci$|automation|actions-user|semantic-release|release-bot|-builder$|auto-merge|mergify|fossabot|houndci", re.I)
HAND_BOTS = {"blsqbot", "framabot", "terraframe-builder", "chiaautomation", "aam-digital-ci", "openelis-b

In [11]:
KEY = ["pr_opened_human", "pr_merged_human", "pr_merged_dep_bot", "commits_human", "release_total", "issues_closed", "core_team_dpg"]
if not LIVE:
    say("run 03_aggregate.py (reads cache/*.json, all_repos.csv, dpg_confidence.csv; no network)")
    dq = pd.read_csv(DATA / "dpg_quarterly.csv")
    print(f"\nFinished file instead: data/dpg_quarterly.csv: {len(dq):,} rows, {dq.dpgid.nunique()} DPGs, quarters {dq.quarter.min()} to {dq.quarter.max()}")
    one = dq[dq.dpgid == DEMO_DPG_ID]
    print(f"\nThe same DPG that demo mode builds ({DEMO_DPG_ID}), its four busiest complete quarters:")
    display(one[~one.partial_quarter].nlargest(4, "pr_merged_human")[["quarter"] + KEY].sort_values("quarter"))
else:
    run_script("03_aggregate.py")
    mine = pd.read_csv(WORK / "data/dpg_quarterly.csv")
    print("\nThe DPG's four busiest complete quarters, built just now (one row per DPG per quarter):")
    display(mine[~mine.partial_quarter].nlargest(4, "pr_merged_human")[["quarter"] + KEY].sort_values("quarter"))
    if MODE == "demo":
        rel = pd.read_csv(DATA / "dpg_quarterly.csv"); rel = rel[rel.dpgid == DEMO_DPG_ID]
        j = mine.merge(rel, on=["dpgid", "quarter"], suffixes=("_new", "_released"))
        j = j[~j.partial_quarter_new]
        print("\nCompared with the released data/dpg_quarterly.csv (same DPG, complete quarters):")
        for col in KEY:
            same = (j[col + "_new"].fillna(0) == j[col + "_released"].fillna(0)).mean()
            print(f"  {col:20s} identical in {same:5.0%} of {len(j)} quarters")
        print("Differences, if any, come from activity after the release snapshot (29 Oct 2026): the demo collects GitHub as it is today.")

[read mode] would: run 03_aggregate.py (reads cache/*.json, all_repos.csv, dpg_confidence.csv; no network)

Finished file instead: data/dpg_quarterly.csv: 3,837 rows, 154 DPGs, quarters 2020Q1 to 2026Q4

The same DPG that demo mode builds (GID0090256), its four busiest complete quarters:


,quarter,pr_opened_human,pr_merged_human,pr_merged_dep_bot,commits_human,release_total,issues_closed,core_team_dpg
1802,2020Q1,6,5,0,8,0,8,2
1803,2020Q2,12,9,0,9,0,1,2
1810,2022Q1,8,6,1,7,0,5,2
1819,2024Q2,4,4,0,5,0,3,1


### Step 6. Status (`03b_status.py`)

In [12]:
if not LIVE:
    run_script("03b_status.py")
    ds = pd.read_csv(DATA / "dpg_status.csv")
    print("\nFinished file instead: data/dpg_status.csv, high-confidence repo set:")
    print(ds[ds.repo_set == "high"].status.value_counts().to_string())
else:
    run_script("03b_status.py")
    display(pd.read_csv(WORK / "data/dpg_status.csv"))

[read mode] would: run  python -I scripts/03b_status.py   (inside the scratch workspace)

Finished file instead: data/dpg_status.csv, high-confidence repo set:
status
active     109
dormant     44
slowing     17


### Step 7. AI features (`03f_ai_feature_prs.py`, `02e_release_notes.py`, `03g_release_ai.py`)

`03f` reads the cache; `02e` then fetches release notes for the DPGs it found (it reads the token from the `.env` file, so it is
run in full mode only), and `03g` checks whether each PR reached a release. In demo mode we run `03f` on the one cached repo. To keep
text out of the notebook output, only counts, dates and the matched keyword are shown.

In [13]:
if not LIVE:
    run_script("03f_ai_feature_prs.py"); run_script("02e_release_notes.py"); run_script("03g_release_ai.py")
    y = pd.read_csv(DATA / "ai_feature_prs_yearly.csv")
    print("\nFinished file instead: data/ai_feature_prs_yearly.csv"); display(y.head(8))
else:
    run_script("03f_ai_feature_prs.py")
    if RUN_FULL:
        run_script("02e_release_notes.py"); run_script("03g_release_ai.py")
    else:
        print("(02e and 03g are only run in full mode)")
    f = WORK / "data/ai_feature_prs.csv"
    mine = pd.read_csv(f)
    print(f"\nAI-feature PRs found: {len(mine)}")
    if len(mine): display(mine[["dpg", "merged_at", "source", "term"]])

[read mode] would: run  python -I scripts/03f_ai_feature_prs.py   (inside the scratch workspace)
[read mode] would: run  python -I scripts/02e_release_notes.py   (inside the scratch workspace)
[read mode] would: run  python -I scripts/03g_release_ai.py   (inside the scratch workspace)

Finished file instead: data/ai_feature_prs_yearly.csv


,confidence,year,merged_prs,ai_feature_prs,ai_feature_prs_title,dpgs,dpgs_with_ai_feature_pr
0,high,2020,30794,47,14,79,8
1,high,2021,31361,50,24,93,13
2,high,2022,34249,24,19,114,8
3,high,2023,42581,39,15,123,16
4,high,2024,42862,101,62,130,18
5,high,2025,43860,281,166,122,37
6,high,2026,50631,578,218,117,51
7,medium,2020,211,0,0,2,0


### Step 8. Review classes and AI-disclosure (`13b_review_ai.py`, `03h_ai_disclosure.py`)

`pr_level.csv` is what the release keeps of these two analyses, one row per merged PR. In read mode we count its classes. In demo mode both scripts are run on the one cached repo (03h needs 03c's term lists, copied with it), which gives tiny numbers but the same tables.

In [14]:
if not LIVE:
    run_script("13b_review_ai.py"); run_script("03h_ai_disclosure.py")
    pl = pd.read_csv(DATA / "pr_level.csv", usecols=["actor_class", "review_class", "ai_disclosure_claude"])
    print(f"\nFinished file instead: data/pr_level.csv: {len(pl):,} merged PRs")
    print("review_class (human PRs merged 2020+):", pl.review_class.value_counts().to_dict())
    print("ai_disclosure_claude:", pl.ai_disclosure_claude.value_counts().to_dict())
    prec = pd.read_csv(DATA / "pr_text_ai_precision.csv")
    print("\nPrecision gate (hand-checked sample), share of matches that really say AI helped write the change:")
    print(prec.groupby("group").verdict.agg(["mean", "size"]).round(2).to_string())
else:
    run_script("13b_review_ai.py"); run_script("03h_ai_disclosure.py")
    ft = WORK / "docs/post/findings_tables"
    for name in ("t7_review_ai_by_year.csv", "t8_ai_disclosure_by_year.csv"):
        if (ft / name).exists():
            print("\n" + name); display(pd.read_csv(ft / name))

[read mode] would: run  python -I scripts/13b_review_ai.py   (inside the scratch workspace)
[read mode] would: run  python -I scripts/03h_ai_disclosure.py   (inside the scratch workspace)

Finished file instead: data/pr_level.csv: 276,525 merged PRs
review_class (human PRs merged 2020+): {'human_reviewed': 194911, 'unreviewed': 67696, 'ai_only': 4184, 'automation_only': 929}
ai_disclosure_claude: {False: 271716, True: 4146}

Precision gate (hand-checked sample), share of matches that really say AI helped write the change:
               mean  size
group                    
chatgpt_codex  0.33    30
claude         0.82    60
copilot        0.57    28
cursor         0.00     1
generic        0.67    30
other_tool     0.17    30


### Step 9. Funding exposure

Most of this step is hand work plus calls to outside APIs, and parts of it **cannot be re-run from the release**:

| part | script | can it run from the release? |
|---|---|---|
| owner clues | `04_owner_clues.py` | yes in full mode (GitHub REST + the DPGs' own websites); but the clues are only the input to a **hand-curated** owner map |
| owner map | `04b_owner_rebuild_d38.py` (withheld: its hand-coded table names an individual) | **no**; the finished `dpg_owner_map.csv` is released |
| EIN candidates, 990 summary | `05_ein_candidates.py`, `06_pull_990.py` (ProPublica) | yes in full mode; the choice of EIN is **hand-curated** (`org_ein_map.csv`) |
| IRS e-file 990 panel | `09_990_panel.py` + a download helper (withheld for privacy: it carried a personal contact address) | **no**; the finished `org_990_panel.csv` is released |
| USAspending awards | `08_usaspending.py`, `08a_build_recipient_map.py` | the recipient map is **hand-reviewed** and not released; the finished `funding_flags.csv` is |
| IATI donors | `11_iati_collect.py`, `11_iati_match.py` | needs an IATI Datastore key; finished `donor_flags.csv` is released |
| terminations | `11_terminations.py` | needs the source workbook (figshare 28881605), not released |
| joint sample, locked test | `07_analysis_sample.py`, `09_event_study_spec.py` | need the unreleased maps above; the outputs `analysis_sample.csv` and `prereg_lock.json` are released |

Below we show the finished funding files and how the evidence columns look.

In [15]:
om  = pd.read_csv(DATA / "dpg_owner_map.csv")
ein = pd.read_csv(DATA / "org_ein_map.csv")
ff  = pd.read_csv(DATA / "funding_flags.csv")
dn  = pd.read_csv(DATA / "donor_flags.csv")
p990 = pd.read_csv(DATA / "org_990_panel.csv")
print("dpg_owner_map.csv   (hand-curated)  rows:", len(om), "| owner types:", om.drop_duplicates("dpgid").owner_type.value_counts().to_dict())
print("org_ein_map.csv     (hand-curated)  rows:", len(ein), "| confidence:", ein.confidence.value_counts().to_dict())
print("funding_flags.csv   exposure_score:", ff.exposure_score.value_counts().sort_index().to_dict() if "exposure_score" in ff else list(ff.columns))
print("donor_flags.csv     terminated_award (non-empty):", int(dn.terminated_award.fillna("").astype(str).str.startswith("yes").sum()) if "terminated_award" in dn else "-")
print("org_990_panel.csv   rows:", len(p990), "| EINs:", p990.ein.nunique() if "ein" in p990 else "-")
if LIVE:
    print("\nOwner clues for the demo DPG would be collected by 04_owner_clues.py (full mode only).")

dpg_owner_map.csv   (hand-curated)  rows: 256 | owner types: {'CO': 68, 'NP': 57, 'USNP': 44, 'GOV': 30, 'UN': 29, 'UNIV': 13, 'COMM': 10, 'UNKNOWN': 3, 'MDB': 2}
org_ein_map.csv     (hand-curated)  rows: 39 | confidence: {'high': 31, 'unmatched': 4, 'low': 2, 'medium': 2}
funding_flags.csv   exposure_score: {0.0: 219, 1.0: 2, 2.0: 6, 3.0: 6}
donor_flags.csv     terminated_award (non-empty): 10
org_990_panel.csv   rows: 247 | EINs: 35


In [16]:
if RUN_FULL:
    # copy the two hand-curated maps next to the scripts so the ProPublica steps can read them (inputs only; outputs go to the scratch folder)
    for f in ("dpg_owner_map.csv", "org_ein_map.csv"): shutil.copy2(DATA / f, WORK / "data" / f)
    run_script("04_owner_clues.py"); run_script("05_ein_candidates.py"); run_script("06_pull_990.py")
else:
    say("run 04_owner_clues.py, 05_ein_candidates.py, 06_pull_990.py in the scratch workspace (full mode only). The other funding scripts need unreleased hand-curated inputs.")

[read mode] would: run 04_owner_clues.py, 05_ein_candidates.py, 06_pull_990.py in the scratch workspace (full mode only). The other funding scripts need unreleased hand-curated inputs.


### Step 10. The build script

`99_build_release.py` is not in `pipeline/` (it lives in the project's `scripts/` folder and needs the project's raw files), so it cannot be run from the release.
Here is what it leaves behind, checked from the finished folder: no e-mail addresses, and person ids that are hashes.

In [17]:
email = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
hits = []
for f in sorted(DATA.glob("*")):
    if f.suffix in (".csv", ".md", ".json") and f.stat().st_size < 30e6:
        t = f.read_text(encoding="utf-8", errors="ignore")
        n = len(email.findall(t))
        if n: hits.append((f.name, n))
print("files in data/ containing an e-mail address:", hits or "none")
pq = pd.read_csv(DATA / "people_quarterly.csv", nrows=3)
print("sample person ids (hashes, not logins):", pq.person_id.tolist())
print("pipeline scripts shipped:", len(list(PIPE.glob('*.py'))), "| withheld by the build script: msg.py, irs_zip.py, git_snapshot.py, 04b_owner_rebuild_d38.py, 99_build_release.py")

files in data/ containing an e-mail address: none
sample person ids (hashes, not logins): ['8b936e810b0d', 'b943feaaada9', '21596eb46527']
pipeline scripts shipped: 37 | withheld by the build script: msg.py, irs_zip.py, git_snapshot.py, 04b_owner_rebuild_d38.py, 99_build_release.py


## Part C. Inventory of `data/`

For every file: rows, columns, the step that produced it, and which of the analysis notebooks 01 to 05 read it. The last column is
computed by searching the notebooks' code for the file name, so it stays correct if a notebook changes.

In [18]:
# Which step and script produced each file. (Written by hand from the walk-through above; a file that is not listed shows '?'.)
PRODUCER = {
 "registry_dpgs.csv": (1, "build script, from the API response"),
 "dpg_repo_map.csv": (2, "01_clean_registry.py"),
 "dpgs_without_code_repo.csv": (2, "01_clean_registry.py"),
 "all_repos.csv": (3, "02a_inventory_repos.py + 02b_confidence_tiers.py (org rows rejected by the name rule dropped by the build)"),
 "all_repos_not_matched_summary.csv": (3, "build script, summarising the dropped org rows"),
 "dpg_confidence.csv": (3, "02b_confidence_tiers.py"),
 "repo_quarterly.csv": (5, "03_aggregate.py"),
 "dpg_quarterly.csv": (5, "03_aggregate.py"),
 "people_quarterly.csv": (5, "03_aggregate.py (ids hashed)"),
 "dpg_no_cohort_repo.csv": (5, "03_aggregate.py"),
 "repo_status.csv": (6, "03b_status.py"),
 "dpg_status.csv": (6, "03b_status.py"),
 "ai_feature_prs.csv": (7, "03f_ai_feature_prs.py (text columns removed)"),
 "ai_feature_prs_shipped.csv": (7, "03g_release_ai.py"),
 "ai_feature_prs_yearly.csv": (7, "03f_ai_feature_prs.py"),
 "release_notes_ai.csv": (7, "02e_release_notes.py + 03g_release_ai.py (text columns removed)"),
 "pr_level.csv": (8, "build script, distilled from the cache with the 13b / 03h rules"),
 "review_bots_by_year.csv": (8, "build script (13b rules)"),
 "pr_text_ai_precision.csv": (8, "hand verdicts on 03c samples, kept by the build script"),
 "qgis_template_check.csv": (8, "build script (03h negation filter evidence)"),
 "dpg_owner_map.csv": (9, "04_owner_clues.py -> HAND-CURATED (04b withheld) -> 08c review"),
 "org_ein_map.csv": (9, "05_ein_candidates.py -> HAND-CURATED"),
 "funding_flags.csv": (9, "08b_funding_flags.py"),
 "donor_flags.csv": (9, "11_donor_flags.py (+ 11_iati_*, 11_join_terminations.py)"),
 "exposed_dpgs_case_list.csv": (9, "08d_exposed_case_list.py"),
 "org_990_panel.csv": (9, "09_990_panel.py (IRS e-file XML; helper withheld)"),
 "org_990_panel_codebook.md": (9, "09_990_panel.py"),
 "analysis_sample.csv": (9, "07_analysis_sample.py"),
 "prereg_lock.json": (9, "09_event_study_spec.py --lock"),
}

# Which notebooks read which file: look for the file name in the code cells of notebooks 01 to 05.
nbs = sorted(p for p in REL.glob("0[1-5]_*.ipynb"))
nb_code = {p.name[:2]: "\n".join("".join(c["source"]) for c in json.loads(p.read_text(encoding="utf-8"))["cells"] if c["cell_type"] == "code") for p in nbs}

def count_rows(p):
    # Small files: let pandas count (some text fields contain line breaks, so counting newlines would over-count).
    # Big files (pr_level, people_quarterly) hold only ids, dates and numbers, so counting newlines in binary chunks is exact and cheap.
    if p.stat().st_size < 20e6:
        return len(pd.read_csv(p, low_memory=False))
    n = 0
    with open(p, "rb") as fh:
        while chunk := fh.read(1 << 20): n += chunk.count(b"\n")
    return max(n - 1, 0)

rows = []
for f in sorted(DATA.iterdir()):
    if not f.is_file(): continue
    if f.suffix == ".csv":
        ncol = len(pd.read_csv(f, nrows=0).columns); nrow = count_rows(f)
    else:
        ncol = nrow = None
    step, how = PRODUCER.get(f.name, ("?", "(not listed in this notebook's table)"))
    used = ",".join(k for k, t in nb_code.items() if f.name in t) or "-"
    rows.append(dict(file=f.name, rows=nrow, columns=ncol, size_kb=round(f.stat().st_size / 1024), step=step, produced_by=how, used_by_notebooks=used))
inv = pd.DataFrame(rows).sort_values(["step", "file"], key=lambda s: s.astype(str) if s.name == "file" else s.map(lambda x: 99 if x == "?" else x)).reset_index(drop=True)
display(inv.style.hide(axis="index").set_properties(**{"text-align": "left"}).format({"rows": "{:,.0f}", "columns": "{:,.0f}", "size_kb": "{:,}"}, na_rep="-"))
print(f"\n{len(inv)} files, {inv.size_kb.sum()/1024:.1f} MB. Files no analysis notebook reads (kept as evidence or for the build): "
      f"{inv[inv.used_by_notebooks == '-'].file.tolist()}")

file,rows,columns,size_kb,step,produced_by,used_by_notebooks
registry_dpgs.csv,256,5,16,1,"build script, from the API response",01
dpg_repo_map.csv,240,11,67,2,01_clean_registry.py,01
dpgs_without_code_repo.csv,54,7,13,2,01_clean_registry.py,01
all_repos.csv,"1,170",19,221,3,02a_inventory_repos.py + 02b_confidence_tiers.py (org rows rejected by the name rule dropped by the build),01
all_repos_not_matched_summary.csv,8,3,0,3,"build script, summarising the dropped org rows",-
dpg_confidence.csv,202,9,11,3,02b_confidence_tiers.py,"01,02,03,05"
dpg_no_cohort_repo.csv,23,6,4,5,03_aggregate.py,01
dpg_quarterly.csv,"3,837",54,691,5,03_aggregate.py,"01,02,05"
people_quarterly.csv,"47,448",6,"2,630",5,03_aggregate.py (ids hashed),02
repo_quarterly.csv,"4,323",49,"1,231",5,03_aggregate.py,02



29 files, 29.0 MB. Files no analysis notebook reads (kept as evidence or for the build): ['all_repos_not_matched_summary.csv', 'repo_status.csv', 'donor_flags.csv', 'dpg_owner_map.csv', 'org_990_panel_codebook.md', 'org_ein_map.csv']


## What cannot be reproduced from the release

* **The raw GitHub cache** (1.9 GB) is not released, so steps 5 to 8 cannot be re-run on the full registry without re-running step 4 (hours, a token). Demo mode shows that they do reproduce the released numbers for a single DPG.
* **Hand-curated files**: `manual_repo_review.csv` (not released; its effect is in `all_repos.csv`), the owner map, the EIN map and the US-awards recipient map (not released). They are judgement calls from public evidence, each with a confidence column.
* **The IRS e-file download helper** was withheld for privacy; the Form 990 panel is released as a finished file.
* **Outside lists**: the ITPC termination workbook and an IATI key are needed to rebuild `donor_flags.csv`.
* **The registry as it was**: step 1 fetches the registry *today*; DPGs certified or lapsed since 29 October 2026 will differ from the release.
* **GitHub moves on**: a fresh collection reflects repositories as they are now, so recent quarters can differ slightly from the released panels.

In [19]:
# Optional clean-up of the demo workspace (set CLEAN_DEMO_AFTER = True at the top). The folder is also in .gitignore.
if MODE == "demo" and CLEAN_DEMO_AFTER and WORK.exists() and WORK.name == "_demo_output":
    shutil.rmtree(WORK); print("removed", WORK)
elif LIVE:
    print("scratch workspace kept at", WORK, "(safe to delete; it is in .gitignore)")
else:
    print("read mode: nothing was written anywhere.")

read mode: nothing was written anywhere.
